In [ ]:
import numpy as np
import pickle
from pathlib import Path
from itertools import product
from joblib import Parallel, delayed
import pandas as pd
from typing import Sequence, Optional

from helpers import create_folder_structure, write_ready_manifest, check_data_availability, load_config, create_minimal_summary
from syllabification import parse_to_phones_and_sylls

from ngram_model import JelinekMercerModel

import logging 
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] [%(process)d] %(message)s'
)
logger = logging.getLogger()

def run_test_pipeline() -> None:
    """
    Execute a lightweight, end-to-end sanity run for multiple (language, unit, text) tasks.

    - Ensures folder structure exists.
    - Prepares inputs if they do not exist yet for a specific corpus size.
    - Runs the pipeline for adjustable corpus sizes.
    - Writes a readiness manifest *once per task* if all required data and preperation is available for the largest corpus size.
    - Produces a minimal summary CSV (if any results are available) for sanity checks.

    Notes
    -----
    - This function is meant for interactive/testing use, not multi-day jobs.
    - Heavy runs should be launched by the separate long-running .py script.
    """

    # Create all combinations to process, adjust as needed
    languages = ['FRA', 'DEU', 'ENG']
    unit_types = ['phones', 'sylls', 'words']
    text_types = ['across_sentences']
    n_values = [1,2]  
    folder_name = "produced_data_large_corpus"
    corpus_size = 100  # 'max' or specific number (int) for testing

    create_folder_structure(languages, folder_name, unit_types)

    tasks = list(product(languages, unit_types, text_types))
    results = Parallel(n_jobs=1, backend="loky", verbose=10)( # Adjust n_jobs based on number of tasks
        delayed(check_ready_to_run)(
            language=lang,
            unit_type=unit,
            text_type=txt,
            n_values=n_values,
            folder_name=folder_name,
            corpus_size=corpus_size,
        )
        for lang, unit, txt in tasks
    )

    # Filter and concatenate any returned DataFrames
    summary_dfs = [res for res in results if isinstance(res, pd.DataFrame)]
    if summary_dfs:
        full_summary = pd.concat(summary_dfs, ignore_index=True)
        create_minimal_summary(full_summary, corpus_size)

    success_count = sum(
        r is not None or (unit == "words" and txt == "within_words")
        for (_, unit, txt), r in zip(tasks, results)
    )

    if success_count == 0:
        logging.error("❌ No valid results. Please check the input data and configurations.")
    else:
        logging.info(f"✅ {success_count} tasks completed successfully.")

def check_ready_to_run(
    language: str,
    unit_type: str,
    text_type: str,
    n_values: Sequence[int],
    folder_name: str,
    corpus_size: int | str,
) -> Optional[pd.DataFrame]:

    # Load language config
    config_dict = load_config(language) 
    folder = str(Path(folder_name) / language)

    input_path = check_data_availability(
        language=language,
        unit_type=unit_type,
        config_dict=config_dict,
        folder_name=folder_name,
        corpus_size=corpus_size,
    )
    if input_path is None:
        logger.error(f"[{language}/{unit_type}/{text_type}] Input not available.")
        return None

    (
        existing_ipa_path,
        phonized_path,
        syllabified_path,
        corpus_size_actual,
        is_near_expected
    ) = parse_to_phones_and_sylls(
        language=language,
        config_dict=config_dict,
        folder=folder,
        corpus_size=corpus_size,
    )

    # Define the input path with unit_type
    if unit_type == 'words': 
        input_path = existing_ipa_path
    else: 
        input_path = phonized_path if unit_type == 'phones' else syllabified_path

    if not input_path.exists():
        logger.error(f"Input path does not exist: {input_path}")
        return None

    # Open the data file
    with open(input_path, "rb") as f:
        data = pickle.load(f)
        
    # Skip invalid combination
    if unit_type == 'words' and text_type == 'within_words':
        return
     
    df_rows = []
    # Compute for each n IR, ID, SR, PPL and best lambdas
    for n in n_values:
        jm_model = JelinekMercerModel(n)
        results = jm_model.fit_with_tuning(data, text_type, unit_type, language)

        logging.info(
        f"[{language} | n={n} | {text_type} | {unit_type}] "
        f"Best λs: {results['best_lambdas']} | "
        f"Mean IR: {np.mean(results['info_rate_values']):.3f} | "
        f"Dev PPL={results['dev_perplexity']:.3f} | "
        f"Test PPL={results['test_perplexity']:.3f}"
        )

        # Display a table with the computed numbers
        metrics = {
        "ID": results["info_density"],
        "IR": np.mean(results["info_rate_values"]),
        "dev_PPL": results["dev_perplexity"],
        'test_PPL': results["test_perplexity"],
        }

        # Add each lambda_k as its own numeric metric
        for k, v in results["best_lambdas"].items():
            metrics[f"lambda_{k}"] = v

        # Append to df_rows in long format (Metric, Value)
        df_rows.extend([
            {
                "Language": language,
                "UnitType": unit_type,
                "TextType": text_type,
                "n": n,
                "Metric": metric,
                "Value": round(value, 6) if isinstance(value, (int, float)) else value
            }
            for metric, value in metrics.items()
        ])

        # free memory
        del jm_model

    # Write manifest if all preprocessing steps are done for the largest corpus size
    if is_near_expected:
        manifest_path = write_ready_manifest(
        language=language,
        unit_type=unit_type,
        folder_name=folder_name,
        existing_ipa_path=existing_ipa_path,
        phonized_path=phonized_path,
        syllabified_path=syllabified_path,
    )

        logging.info(f"✅ Created manifest: task [{language}, {unit_type}, {text_type}] is ready to run via the IR pipeline.")
    else: 
        logging.info(f"Manifest not written due to small corpus size ({corpus_size_actual}). Use corpus_size='max' if you want to use the full corpus.")

    return pd.DataFrame(df_rows) if df_rows else None

if __name__ == "__main__":
    run_test_pipeline() 

In [ ]:
from helpers import clean_corpus_size_files
# Clean up any files from previous test runs 
# Specify the parameters as needed
clean_corpus_size_files(
    base_folder='test_runs',
    language_codes=['FRA', 'DEU', 'ENG'],
    corpus_size=100,
    unit_types=['phones', 'sylls', 'words']
)
